In [1]:
import os
import warnings
import logging
import tensorflow as tf

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["ABSL_LOG_LEVEL"] = "3"

warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")
logging.getLogger("tensorflow").setLevel(logging.ERROR)

import sys
try:
    dataset_name = os.listdir('/kaggle/input/datasets/keithmarange')[0]
    sys.path.append(f'/kaggle/input/datasets/keithmarange/{dataset_name}/')
    sys.path.append('/kaggle/input/cmi-competition-code')
except Exception as e:
    pass

import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from scipy.spatial.transform import Rotation as R

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, GridSearchCV, GroupShuffleSplit

from sklearn.metrics import f1_score, classification_report

import data_utils
from base_utils_qwen import SequenceExtractor, prepare_multitask_param_space, prepare_bayesian_space
from proto_utils_qwen import SingleHeadPrototypicalNetwork, MultiHeadPrototypicalNetwork, competition_scorer
try:
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True
except ImportError:
    SKOPT_AVAILABLE = False

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

2026-06-09 02:33:53.479675: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780972433.962169      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780972434.102153      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780972435.351775      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780972435.351820      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780972435.351823      23 computation_placer.cc:177] computation placer alr

In [2]:
# ============================================================
# CONFIGURATION - CHANGE THIS TO SWITCH MODES
# ============================================================

# Set this to "single" or "multi"
mode = "single"  # "single" for binary+gesture, "multi" for multi-head

single_target = "bfrb"  # For single mode: "gesture", "gesture_action", etc.

# For MULTI mode: which heads to predict
multi_heads = ["gesture_action", "orientation", "gesture_position"]
primary_target = "bfrb"  # For multi mode: what to evaluate F1 on

pipe_name = "extractor"
proto_name = "model"

search_mode = "bayesian"  # "grid" or "bayesian"
random_state = 42
n_splits = 2
n_iter = 10
n_jobs = 1
train_size = 0.4
error_score_constant = np.nan
verbose = 1
do_cross_val = False

if do_cross_val:
    cv_object = GroupKFold(n_splits=n_splits)
else:
    cv_object = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=random_state)  # ✅ Fast

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

print(f"Mode: {mode}")
print(f"Search mode: {search_mode}")

Mode: single
Search mode: bayesian


In [3]:
data_root = data_utils.find_data_root()

raw_train_df = pd.read_csv(data_root / "train.csv")
raw_test_df = pd.read_csv(data_root / "test.csv")
train_demo_df = pd.read_csv(data_root / "train_demographics.csv")
test_demo_df = pd.read_csv(data_root / "test_demographics.csv")

Using Kaggle data folder: /kaggle/input/competitions/cmi-detect-behavior-with-sensor-data


In [4]:
train_df = raw_train_df.set_index("row_id").copy(deep=True)

# Handedness correction
train_df["handedness"] = train_df["subject"].map(train_demo_df.set_index("subject")["handedness"])
left_handed_mask = train_df["handedness"].eq(0)
train_df.loc[left_handed_mask, "acc_x"] *= -1.0

rot_cols = ["rot_w", "rot_x", "rot_y", "rot_z"]
q_wxyz = train_df.loc[left_handed_mask, rot_cols].to_numpy(dtype=float)
q_wxyz = np.nan_to_num(q_wxyz, nan=0.0, posinf=0.0, neginf=0.0)
norm = np.linalg.norm(q_wxyz, axis=1, keepdims=True)
bad_q = norm.squeeze() == 0
q_wxyz[bad_q] = np.array([1.0, 0.0, 0.0, 0.0])
norm = np.linalg.norm(q_wxyz, axis=1, keepdims=True)
q_wxyz = q_wxyz / norm

q_xyzw = q_wxyz[:, [1, 2, 3, 0]]
euler_xyz = R.from_quat(q_xyzw).as_euler("xyz", degrees=False)
euler_xyz[:, [1, 2]] *= -1.0
q_xyzw_fixed = R.from_euler("xyz", euler_xyz, degrees=False).as_quat()
q_wxyz_fixed = q_xyzw_fixed[:, [3, 0, 1, 2]]
train_df.loc[left_handed_mask, rot_cols] = q_wxyz_fixed

print(f"left-handed corrected: {left_handed_mask.sum()} rows")

# Upside-down correction
upside_down_mask = train_df["subject"].isin(["SUBJ_019262", "SUBJ_045235"])
train_df.loc[upside_down_mask, ["acc_x", "acc_y", "acc_z"]] *= -1.0
train_df.loc[upside_down_mask, ["rot_x", "rot_y", "rot_z"]] *= -1.0
print(f"upside-down corrected: {upside_down_mask.sum()} rows")

train_df = train_df.drop(columns=["handedness"])

left-handed corrected: 71352 rows
upside-down corrected: 12257 rows


In [5]:
# Create target columns
train_df["gesture_position"] = train_df["gesture"].str.split(" - ").str[0]
train_df["gesture_action"] = train_df["gesture"].str.split(" - ").str[-1]
train_df['is_target'] = (train_df['sequence_type'] == 'Target').astype(bool)
train_df['bfrb'] = train_df['gesture'].where(train_df['is_target'], 'non_bfrb')

train_sample_df, hold_out_df = data_utils.sample_balanced_split(
    train_df, train_pct=train_size, test_pct=(1 - train_size), random_state=random_state
)

print(f"Train sequences: {train_sample_df['sequence_id'].nunique()}")
print(f"Hold Out sequences: {hold_out_df['sequence_id'].nunique()}")

Train: 2910 seqs | 35.7%
Test:  2881 seqs  | 35.3%
Train sequences: 2910
Hold Out sequences: 2881


In [6]:
train_sample_df.groupby(['bfrb', 'orientation']).agg(
    {'sequence_id':'nunique'})

sequence_id
bfrb                     orientation                                 
Above ear - pull hair    Lie on Back                               40
                         Lie on Side - Non Dominant                37
                         Seated Lean Non Dom - FACE DOWN           45
                         Seated Straight                           40
Cheek - pinch skin       Lie on Back                               37
                         Lie on Side - Non Dominant                45
                         Seated Lean Non Dom - FACE DOWN           43
                         Seated Straight                           37
Eyebrow - pull hair      Lie on Back                               30
                         Lie on Side - Non Dominant                38
                         Seated Lean Non Dom - FACE DOWN           46
                         Seated Straight                           48
Eyelash - pull hair      Lie on Back                               41
                         Lie on Side - Non Dominant                39
                         Seated Lean Non Dom - FACE DOWN           48
                         Seated Straight                           34
Forehead - pull hairline Lie on Back                               32
                         Lie on Side - Non Dominant                44
                         Seated Lean Non Dom - FACE DOWN           46
                         Seated Straight                           40
Forehead - scratch       Lie on Back                               42
                         Lie on Side - Non Dominant                44
                         Seated Lean Non Dom - FACE DOWN           44
                         Seated Straight                           32
Neck - pinch skin        Lie on Back                               42
                         Lie on Side - Non Dominant                40
                         Seated Lean Non Dom - FACE DOWN           30
                         Seated Straight                           50
Neck - scratch           Lie on Back                               41
                         Lie on Side - Non Dominant                46
                         Seated Lean Non Dom - FACE DOWN           31
                         Seated Straight                           44
non_bfrb                 Lie on Back                              188
                         Lie on Side - Non Dominant               201
                         Seated Lean Non Dom - FACE DOWN          529
                         Seated Straight                          696

In [7]:
sequence_extractor = SequenceExtractor(
    acc_modes='raw'
)

if mode == "single":
    # BinaryPlusGesturePrototypicalNetwork uses is_target and gesture automatically
    model = SingleHeadPrototypicalNetwork(
        target=single_target,
    )
else:
    model = MultiHeadPrototypicalNetwork(
        primary_target=primary_target,
    )

pipeline = Pipeline([
    ("extractor", sequence_extractor),
    ("model", model),
])

In [8]:
# ============================================================================
# PARAMETER SPACE DEFINITION
# ============================================================================
if search_mode == "bayesian":
    # ============================================================
    # 1. EXTRACTOR PARAMETERS (AdvancedMultiDomainSequenceExtractor)
    # ============================================================
    base_extractor_params = {
        f"{pipe_name}__acc_modes": Categorical([
            "raw",
            "smoothed"
        ]),
        f"{pipe_name}__rotation_modes": Categorical([
            "quaternion|delta_euler"
        ]),
        f"{pipe_name}__tof_modes": Categorical(["sensor_stats|pooled_diff"]),
        f"{pipe_name}__thm_modes": Categorical(["centered_diff"]),
        
        f"{pipe_name}__motion_filter_mode": Categorical(["kalman"]),
        f"{pipe_name}__kalman_process_noise": Real(1e-4, 1e-2, prior="log-uniform"),
        f"{pipe_name}__kalman_measurement_noise": Real(1e-2, 5e-1, prior="log-uniform"),
        f"{pipe_name}__use_dead_reckoning": Categorical([True, False]),
        f"{pipe_name}__dead_reckoning_detrend": Categorical([True, False]),
        
        f"{pipe_name}__sampling_rate": Categorical([20, 50, 100]),
        f"{pipe_name}__compute_dt": Categorical([True]),
        f"{pipe_name}__clip_value": Categorical([50.0]),
        f"{pipe_name}__interp_mode": Categorical(["linear"]),
        f"{pipe_name}__window_size": Categorical([50]),
        f"{pipe_name}__smooth_alpha": Categorical([0.652]),
        f"{pipe_name}__maxlen": Categorical([200]),
        f"{pipe_name}__padding_value": Categorical([-999.0]),
    }
    
    # ============================================================
    # 2. MODEL PARAMETERS (Proto Utils: Single/Multi Head)
    # ============================================================
    base_model_params = {
        f"{proto_name}__backbone_type": Categorical(["1dcnn"]),
        f"{proto_name}__filters": Categorical(["128-256-256", "256-512-512", "256-512-512-512"]),
        f"{proto_name}__kernels": Categorical(["3-3", "5-3", "7-5-3"]),
        f"{proto_name}__pools": Categorical(["none", "2", "2-2"]),
        
        f"{proto_name}__dropout": Real(0.4, 0.55),
        f"{proto_name}__embed_dim": Categorical([256, 512]),
        # f"{proto_name}__lstm_units": Integer(64, 256),
        
        f"{proto_name}__learning_rate": Categorical([5e-3]),
        f"{proto_name}__batch_size": Categorical([16]),
        f"{proto_name}__epochs": Categorical([50]),
        f"{proto_name}__patience": Categorical([15]),
        
        f"{proto_name}__n_way": Categorical([9]), 
        f"{proto_name}__n_support": Categorical([80]),
        f"{proto_name}__n_query": Categorical([60, 70, 80]),
        
        f"{proto_name}__use_mixup": Categorical([True]),
        f"{proto_name}__mixup_alpha": Real(0.1, 0.6),
        f"{proto_name}__use_time_mask": Categorical([True, False]),
        f"{proto_name}__time_mask_ratio": Real(0.05, 0.2),
        f"{proto_name}__use_gaussian_noise": Categorical([True, False]),
        f"{proto_name}__noise_std": Real(0.001, 0.05, prior="log-uniform"),
    }
    
    # ============================================================
    # 3. MODE‑SPECIFIC PARAMETERS
    # ============================================================
    if mode == "single":
        param_space = {
            **base_extractor_params,
            **base_model_params,
            f"{proto_name}__target": Categorical([single_target]),
        }
    else:  # multi mode
        heads_tuple = tuple(multi_heads)
        param_space = {
            **base_extractor_params,
            **base_model_params,
            f"{proto_name}__sub_heads": Categorical([heads_tuple]),
            f"{proto_name}__primary_target": Categorical([primary_target]),
        }
    
    # ============================================================
    # 4. CONVERT NON‑SCALARS TO JSON STRINGS (FOR BAYESIAN SEARCH)
    # ============================================================
    param_space = prepare_bayesian_space(param_space)


else:  # Grid Search (your original settings, unchanged)
    param_space = {
        # ============================================================
        # 1. EXTRACTOR PARAMETERS (AdvancedMultiDomainSequenceExtractor)
        # ============================================================
        f"{pipe_name}__acc_modes": ["raw|jerk"],
        f"{pipe_name}__rotation_modes": ["quaternion|rot6d|angular_velocity"],
        f"{pipe_name}__tof_modes": ["sensor_stats|pooled_stats"],
        f"{pipe_name}__thm_modes": ["centered_diff"],
        
        f"{pipe_name}__motion_filter_mode": ["extended_kalman"],
        f"{pipe_name}__kalman_process_noise": [1e-3],
        f"{pipe_name}__kalman_measurement_noise": [1e-1],
        f"{pipe_name}__use_dead_reckoning": [True],
        f"{pipe_name}__dead_reckoning_detrend": [True],
        
        f"{pipe_name}__sampling_rate": [100],
        f"{pipe_name}__compute_dt": [True],
        f"{pipe_name}__clip_value": [50.0],
        f"{pipe_name}__interp_mode": ["linear"],
        f"{pipe_name}__window_size": [20],
        f"{pipe_name}__smooth_alpha": [0.5],
        f"{pipe_name}__maxlen": [160],
        f"{pipe_name}__padding_value": [-999.0],
        
        # ============================================================
        # 2. MODEL PARAMETERS (Proto Utils)
        # ============================================================
        f"{proto_name}__backbone_type": ["1dcnn"],
        f"{proto_name}__filters": ["64-64"],
        f"{proto_name}__kernels": ["3-3"],
        f"{proto_name}__pools": ["2"],
        
        f"{proto_name}__dropout": [0.4],
        f"{proto_name}__embed_dim": [64],
        f"{proto_name}__lstm_units": [128],
        
        f"{proto_name}__learning_rate": [1e-3],
        f"{proto_name}__batch_size": [16],
        f"{proto_name}__epochs": [100],
        f"{proto_name}__patience": [15],
        
        f"{proto_name}__n_way": [9],
        f"{proto_name}__n_support": [40],
        f"{proto_name}__n_query": [40],
        
        f"{proto_name}__use_mixup": [False],
        f"{proto_name}__mixup_alpha": [0.4],
        f"{proto_name}__use_time_mask": [False],
        f"{proto_name}__time_mask_ratio": [0.1],
        f"{proto_name}__use_gaussian_noise": [True],
        f"{proto_name}__noise_std": [0.01],
    }
    
    # ============================================================
    # 3. MODE‑SPECIFIC TARGETS (GRID SEARCH)
    # ============================================================
    if mode == "single":
        param_space[f"{proto_name}__target"] = [single_target]
    else:  # multi mode
        param_space[f"{proto_name}__primary_target"] = [primary_target]
        param_space[f"{proto_name}__sub_heads"] = [tuple(multi_heads)]

In [9]:
if mode == "single":
    y_train = train_sample_df[["sequence_id", "is_target", single_target]].copy()
    y_test = hold_out_df[["sequence_id", "is_target", single_target]].copy()
else:
    # For multi mode, include all heads plus the primary target for lookup
    cols = ["sequence_id"] + multi_heads
    if primary_target not in cols:
        cols.append(primary_target)
    y_train = train_sample_df[cols].copy()
    y_test = hold_out_df[["sequence_id", primary_target]].copy()

X_train = train_sample_df.copy()
X_test = hold_out_df.copy()
groups = X_train["sequence_id"]

print(f"MODE: {mode}")
print(f"Single target: {single_target if mode=='single' else 'N/A'}")
print(f"Multi heads: {multi_heads if mode=='multi' else 'N/A'}")
print(f"Primary target: {primary_target if mode=='multi' else 'N/A'}")

print(f"y_train columns: {y_train.columns.tolist()}")

MODE: single
Single target: bfrb
Multi heads: N/A
Primary target: N/A
y_train columns: ['sequence_id', 'is_target', 'bfrb']


In [10]:
if search_mode == "bayesian":

    param_space = prepare_multitask_param_space(param_space, search_mode)
    
    search = BayesSearchCV(
        estimator=pipeline,
        search_spaces=param_space,
        n_iter=n_iter,
        scoring=competition_scorer,
        cv=cv_object,
        n_jobs=n_jobs,
        random_state=random_state,
        verbose=verbose,
        refit=True,
        return_train_score=True,
        error_score=error_score_constant
    )
else:
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_space,
        scoring=competition_scorer,
        cv=cv_object,
        n_jobs=n_jobs,
        verbose=verbose,
        refit=True,
        return_train_score=True,
        error_score=error_score_constant,
    )

print(f"Running {search_mode} search...")
search.fit(X_train, y_train, groups=groups)

print(f"\nBest CV competition score: {search.best_score_:.4f}")
print("Best parameters:")
print(search.best_params_)

best_model = search.best_estimator_

Running bayesian search...
Fitting 1 folds for each of 1 candidates, totalling 1 fits


I0000 00:00:1780972524.523483      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1780972524.529528      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Training for up to 50 epochs (131 steps/epoch, 2096 train / 232 val sequences)...


I0000 00:00:1780972526.759963      23 cuda_dnn.cc:529] Loaded cuDNN version 91002


Epoch   1/50  train_loss=1.6688  train_acc=0.3771  val_loss=1.6987  val_acc=0.3944
Epoch   2/50  train_loss=1.3508  train_acc=0.4925  val_loss=1.5771  val_acc=0.3933
Epoch   3/50  train_loss=1.1609  train_acc=0.5618  val_loss=1.4639  val_acc=0.4728
Epoch   4/50  train_loss=0.9730  train_acc=0.6328  val_loss=1.6475  val_acc=0.4180
Epoch   5/50  train_loss=0.7882  train_acc=0.7079  val_loss=1.5615  val_acc=0.4091
Epoch   6/50  train_loss=0.6104  train_acc=0.7847  val_loss=1.5248  val_acc=0.4598
Epoch   7/50  train_loss=0.4588  train_acc=0.8476  val_loss=1.5642  val_acc=0.5022
Epoch   8/50  train_loss=0.3419  train_acc=0.8887  val_loss=1.5678  val_acc=0.4665
Epoch   9/50  train_loss=0.2695  train_acc=0.9150  val_loss=1.5698  val_acc=0.5046
Epoch  10/50  train_loss=0.2149  train_acc=0.9332  val_loss=1.5929  val_acc=0.4743
Epoch  11/50  train_loss=0.1832  train_acc=0.9428  val_loss=1.4838  val_acc=0.5206
Epoch  12/50  train_loss=0.1520  train_acc=0.9526  val_loss=1.5197  val_acc=0.4860
Epoc

I0000 00:00:1780978542.655960      68 service.cc:152] XLA service 0x68aaf810 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1780978542.656030      68 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1780978542.656040      68 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1780978543.916994      68 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Fitting 1 folds for each of 1 candidates, totalling 1 fits
Training for up to 50 epochs (131 steps/epoch, 2096 train / 232 val sequences)...
Epoch   1/50  train_loss=1.5249  train_acc=0.4299  val_loss=1.4833  val_acc=0.4438
Epoch   2/50  train_loss=1.0544  train_acc=0.5993  val_loss=1.5229  val_acc=0.4567
Epoch   3/50  train_loss=0.7790  train_acc=0.7043  val_loss=1.3767  val_acc=0.4620
Epoch   4/50  train_loss=0.5803  train_acc=0.7884  val_loss=1.5319  val_acc=0.4868
Epoch   5/50  train_loss=0.4503  train_acc=0.8413  val_loss=1.4074  val_acc=0.4945
Epoch   6/50  train_loss=0.3675  train_acc=0.8758  val_loss=1.4745  val_acc=0.5573
Epoch   7/50  train_loss=0.3301  train_acc=0.8897  val_loss=1.4637  val_acc=0.5227
Epoch   8/50  train_loss=0.2797  train_acc=0.9080  val_loss=1.5702  val_acc=0.5115
Epoch   9/50  train_loss=0.2403  train_acc=0.9208  val_loss=1.5116  val_acc=0.5709
Epoch  10/50  train_loss=0.2098  train_acc=0.9316  val_loss=1.5737  val_acc=0.5378
Epoch  11/50  train_loss=0.19

2026-06-09 08:00:45.551894: E external/local_xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng12{k11=2} for conv %cudnn-conv-bias-activation.9 = (f32[32,128,1,200]{3,2,1,0}, u8[0]{0}) custom-call(f32[32,111,1,200]{3,2,1,0} %bitcast.387, f32[128,111,1,7]{3,2,1,0} %bitcast.391, f32[128]{0} %bitcast.393), window={size=1x7 pad=0_0x3_3}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBiasActivationForward", metadata={op_type="Conv2D" op_name="1dcnn_encoder_1/conv1d_1/convolution" source_file="/usr/local/lib/python3.12/dist-packages/tensorflow/python/framework/ops.py" source_line=1200}, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"conv_result_scale":1,"activation_mode":"kRelu","side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false} is taking a while...
2026-06-09 08:00:46.124223: E external/local_xla/xla/service/slow_operation_alarm.cc:140] The operation took 1.572448507s
Trying algorithm e

Fitting 1 folds for each of 1 candidates, totalling 1 fits
Training for up to 50 epochs (131 steps/epoch, 2096 train / 232 val sequences)...
Epoch   1/50  train_loss=1.4504  train_acc=0.4539  val_loss=2.1044  val_acc=0.3765
Epoch   2/50  train_loss=1.0499  train_acc=0.5994  val_loss=2.1294  val_acc=0.3810
Epoch   3/50  train_loss=0.8191  train_acc=0.6913  val_loss=1.4117  val_acc=0.4837
Epoch   4/50  train_loss=0.5984  train_acc=0.7805  val_loss=1.4900  val_acc=0.4934
Epoch   5/50  train_loss=0.4531  train_acc=0.8423  val_loss=1.4691  val_acc=0.5196
Epoch   6/50  train_loss=0.3547  train_acc=0.8801  val_loss=1.3825  val_acc=0.5105
Epoch   7/50  train_loss=0.2798  train_acc=0.9076  val_loss=1.6366  val_acc=0.4795
Epoch   8/50  train_loss=0.2428  train_acc=0.9221  val_loss=1.6902  val_acc=0.4968
Epoch   9/50  train_loss=0.2206  train_acc=0.9289  val_loss=1.6109  val_acc=0.4818
Epoch  10/50  train_loss=0.1985  train_acc=0.9367  val_loss=1.5761  val_acc=0.5292
Epoch  11/50  train_loss=0.17

In [11]:
# ============================================================
# SAVE CV RESULTS TO CSV
# ============================================================

# Convert search results to DataFrame
cv_results_df = pd.DataFrame(search.cv_results_)

# Add metadata columns
cv_results_df['mode'] = mode
cv_results_df['search_mode'] = search_mode
cv_results_df['timestamp'] = timestamp

# Define save path
results_path = results_dir / f"cv_results_{mode}_{search_mode}_{timestamp}.csv"

# Save to CSV
cv_results_df.to_csv(results_path, index=False)

print(f"CV results saved to: {results_path}")
print(f"Shape: {cv_results_df.shape}")
print(f"Columns: {cv_results_df.columns.tolist()}")

CV results saved to: results/cv_results_single_bayesian_20260609_0234.csv
Shape: (10, 53)
Columns: ['mean_fit_time', 'std_fit_time', 'mean_score_time', 'std_score_time', 'param_extractor__acc_modes', 'param_extractor__clip_value', 'param_extractor__compute_dt', 'param_extractor__dead_reckoning_detrend', 'param_extractor__interp_mode', 'param_extractor__kalman_measurement_noise', 'param_extractor__kalman_process_noise', 'param_extractor__maxlen', 'param_extractor__motion_filter_mode', 'param_extractor__padding_value', 'param_extractor__rotation_modes', 'param_extractor__sampling_rate', 'param_extractor__smooth_alpha', 'param_extractor__thm_modes', 'param_extractor__tof_modes', 'param_extractor__use_dead_reckoning', 'param_extractor__window_size', 'param_model__backbone_type', 'param_model__batch_size', 'param_model__dropout', 'param_model__embed_dim', 'param_model__epochs', 'param_model__filters', 'param_model__kernels', 'param_model__learning_rate', 'param_model__mixup_alpha', 'param_m

In [12]:
# ============================================================
# EVALUATE ON HOLDOUT TEST SET - WORKS FOR BOTH MODES
# ============================================================

# Prepare test labels - USE BFRB COLUMN (it has 'non_bfrb' + gestures)
y_test_true = hold_out_df[["sequence_id", "is_target", "bfrb"]].copy()

# Predict (Returns 1 prediction per sequence -> length 4333)
y_pred = best_model.predict(X_test)

# IMPORTANT: Aggregate y_test_true to the sequence level to match y_pred!
# SequenceExtractor groups by sequence_id and sorts them alphabetically/numerically.
# We must do the exact same thing to our ground truth labels.
y_test_true_seq = (
    y_test_true
    .drop_duplicates(subset=["sequence_id"])
    .sort_values("sequence_id")
    .reset_index(drop=True)
)

# Binary F1 (target vs non-target)
y_true_binary = y_test_true_seq["is_target"].values.astype(int)
y_pred_binary = (y_pred != "non_bfrb").astype(int)
binary_f1 = f1_score(y_true_binary, y_pred_binary)

# Gesture Macro F1 (only target/BFRB sequences)
target_mask = y_true_binary == 1
if target_mask.sum() > 0:
    gesture_f1 = f1_score(
        y_test_true_seq.loc[target_mask, "bfrb"].values, 
        y_pred[target_mask], 
        average="macro"
    )
else:
    gesture_f1 = 0.0

competition_score = (binary_f1 + gesture_f1) / 2

print("\n" + "="*60)
print("FINAL EVALUATION")
print("="*60)
print(f"Binary F1 (non_bfrb vs bfrb): {binary_f1:.4f}")
print(f"BFRB Gesture Macro F1: {gesture_f1:.4f}")
print(f"COMPETITION SCORE: {competition_score:.4f}")

if target_mask.sum() > 0:
    print("\n" + "-"*40)
    print("BFRB Gesture Classification Report")
    print("-"*40)
    print(classification_report(
        y_test_true_seq.loc[target_mask, "bfrb"].values, 
        y_pred[target_mask]
    ))

# Save results (Now using sequence-level arrays so lengths match perfectly)
holdout_results = pd.DataFrame({
    "sequence_id": y_test_true_seq["sequence_id"].values,
    "is_target_true": y_true_binary,
    "is_target_pred": y_pred_binary,
    "bfrb_true": y_test_true_seq["bfrb"].values,
    "bfrb_pred": y_pred,
})
holdout_results_path = results_dir / f"holdout_predictions_{timestamp}.csv"
holdout_results.to_csv(holdout_results_path, index=False)
print(f"\nHoldout predictions saved to: {holdout_results_path}")


FINAL EVALUATION
Binary F1 (non_bfrb vs bfrb): 0.9459
BFRB Gesture Macro F1: 0.4015
COMPETITION SCORE: 0.6737

----------------------------------------
BFRB Gesture Classification Report
----------------------------------------
                          precision    recall  f1-score   support

   Above ear - pull hair       0.66      0.52      0.58       241
      Cheek - pinch skin       0.46      0.48      0.47       241
     Eyebrow - pull hair       0.33      0.27      0.30       241
     Eyelash - pull hair       0.40      0.32      0.36       241
Forehead - pull hairline       0.57      0.43      0.49       241
      Forehead - scratch       0.56      0.53      0.55       241
       Neck - pinch skin       0.40      0.41      0.41       241
          Neck - scratch       0.41      0.54      0.46       241
                non_bfrb       0.00      0.00      0.00         0

                accuracy                           0.44      1928
               macro avg       0.42      0.